# 7-4 DataLoader and Reproducible Split — Advanced Practice

강의 원문 대신 직접 작성한 코드, 실행 결과와 학습 메모를 정리했습니다.


In [1]:
# 검증 가능 정답 코드
# split용 Generator를 같은 seed로 각각 새로 만들어 이전 난수 소비가 분할 결과에 섞이지 않게 합니다.
import torch
from torch.utils.data import TensorDataset, random_split

dataset = TensorDataset(torch.arange(12))

def make_split(seed):
    generator = torch.Generator().manual_seed(seed)
    return random_split(dataset, [8, 2, 2], generator=generator)

run1 = make_split(7)
run2 = make_split(7)
reproducible = all(a.indices == b.indices for a, b in zip(run1, run2))
sets = [set(part.indices) for part in run1]
disjoint = not (sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2])
covered = len(set().union(*sets)) == len(dataset)

# 재현성뿐 아니라 교집합 0과 원본 index 전체 합집합까지 확인해 중복·누락을 동시에 잡습니다.
print(f"reproducible={reproducible}")
print(f"disjoint={disjoint}")
print(f"full_coverage={covered}")

reproducible=True
disjoint=True
full_coverage=True


In [2]:
# 검증 가능 정답 코드
# 분할 seed와 train shuffle seed를 분리해 데이터 소속과 epoch 방문 순서를 독립적으로 설명합니다.
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split

dataset = TensorDataset(torch.arange(12).float().reshape(-1, 1), torch.arange(12) % 2)
train_ds, valid_ds, test_ds = random_split(
    dataset, [8, 2, 2], generator=torch.Generator().manual_seed(11)
)
train_loader = DataLoader(
    train_ds, batch_size=3, shuffle=True,
    generator=torch.Generator().manual_seed(22), drop_last=False
)
valid_loader = DataLoader(valid_ds, batch_size=2, shuffle=False, drop_last=False)
test_loader = DataLoader(test_ds, batch_size=2, shuffle=False, drop_last=False)

# 각 split의 sample 수와 loader batch 수를 출력해 마지막 작은 batch가 보존되는지 검산합니다.
print(f"train_samples={len(train_ds)}, batches={len(train_loader)}")
print(f"valid_samples={len(valid_ds)}, batches={len(valid_loader)}")
print(f"test_samples={len(test_ds)}, batches={len(test_loader)}")

train_samples=8, batches=3
valid_samples=2, batches=1
test_samples=2, batches=1


In [3]:
# 검증 가능 정답 코드
# 각 evaluation batch에서 실제로 본 sample ID를 모아 drop_last가 모집단을 줄이는지 직접 측정합니다.
import torch
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(torch.arange(10).reshape(-1, 1))

def seen_samples(drop_last):
    loader = DataLoader(dataset, batch_size=4, shuffle=False, drop_last=drop_last)
    return sum(batch[0].shape[0] for batch in loader)

seen_a = seen_samples(True)
seen_b = seen_samples(False)
# seen과 missing을 후보별로 나란히 남겨 같은 validation 10건을 평가한 구현만 승인합니다.
print(f"candidate_A_seen={seen_a}, missing={len(dataset) - seen_a}")
print(f"candidate_B_seen={seen_b}, missing={len(dataset) - seen_b}")
print("approved=B")

candidate_A_seen=8, missing=2
candidate_B_seen=10, missing=0
approved=B
